# Stockformer: Complete Pipeline — From Raw OHLCV to Stock Predictions

This notebook walks through every step of the **original Stockformer paper** implementation, explaining *what* each step does, *why* it's needed, and giving concrete examples.

**Paper:** Ma, Bohan et al. (2025) — *"Stockformer: A price-volume factor stock selection model based on wavelet transform and multi-task self-attention networks"*, Expert Systems with Applications.

---

### Pipeline at a Glance

```
Step 1 ─ Collect raw OHLCV + fundamental data for Chinese A-shares
  │
Step 2 ─ Clean the stock pool (remove ST, new IPOs, negative NAV)
  │
Step 3 ─ Construct 360 "Alpha" factors from price & volume using Qlib
  │
Step 4 ─ Neutralise factors (remove industry & market-cap biases)
  │
Step 5 ─ Slice into rolling 2-year train / 4-month val / 4-month test subsets
  │
Step 6 ─ Create model inputs: returns (flow), trend indicator, graph embeddings
  │
Step 7 ─ Wavelet Transform decomposes each time-series into Low & High frequency
  │
Step 8 ─ Dual-Encoder Transformer processes Low & High paths separately
  │
Step 9 ─ Adaptive Fusion merges the two frequency representations
  │
Step 10 ─ Multi-Task Heads predict (a) direction (up/down) and (b) return magnitude
```

---
## Step 1 — Collecting Raw OHLCV + Fundamental Data

**Script:** `data_processing_script/volume_and_price_factor_construction/1_stock_data_consolidation.ipynb`

### What happens?
The raw data comes from Chinese financial databases (e.g. CSMAR/Wind). For each trading day and each stock, the following data is collected:

| Category | Example columns | Count |
|----------|----------------|-------|
| **Price** | Open, High, Low, Close (adjusted) | 4 |
| **Volume** | Trading volume, Trading value (money) | 2 |
| **Fundamentals** | PE, PB, ROE, EPS, Net assets per share, etc. | ~25 |
| **Market** | Daily return, Turnover rate, Market-cap weighted avg return | ~8 |

### Concrete example

On **2021-02-01**, stock **000001.SZ** (Ping An Bank) has:

```
Open  = 21.88    High  = 23.78    Low = 21.60    Close = 23.36
Volume = 147,523,930 shares    Money = ¥3.53 billion
PE = ...    PB = ...    Turnover = 0.76%
```

This is stored in a consolidated DataFrame with ~3.1 million rows, covering ~5,000 stocks from the Shanghai (SH), Shenzhen (SZ), and Beijing (BJ) exchanges over 2018–2024.

### Why?
This raw data is the foundation. You cannot compute technical factors (moving averages, RSI, etc.) without first having clean, consistent OHLCV series per stock.

---
## Step 2 — Cleaning the Stock Pool

**Script:** `data_processing_script/volume_and_price_factor_construction/2_data_preprocessing.ipynb`

### What happens?
Not all stocks are suitable for a quantitative model. The paper applies three filters:

| Filter | Rule | Why |
|--------|------|-----|
| **IPO age** | Remove stocks listed < 1 year | New IPOs have abnormal price behaviour |
| **ST status** | Remove stocks ever flagged as ST ("Special Treatment") | ST = financial distress, trading limits differ |
| **Negative NAV** | Remove stocks with negative net asset value | Signals extreme financial distress |

Additionally, the paper only keeps stocks in the **CSI-300 index** (top 300 liquid stocks). After filtering, **~255 stocks** remain.

### Data cleaning steps (per stock)

1. **Outlier winsorisation (MAD method):**
   - Compute median and median absolute deviation (MAD) for each column
   - Cap values at `median ± 3 × MAD`
   - *Example:* If Close prices have median=25, MAD=2, then any value above 31 or below 19 is capped

2. **Missing value imputation:** Forward-fill (`ffill`) — use the most recent known value

3. **Z-score standardisation:** For each stock individually:
   ```
   z = (value − mean) / std_dev
   ```
   This makes all features comparable regardless of their original scale.

### Output
A clean DataFrame with columns: `date, stock_code, open, high, low, close, factor, change, volume, money, float_shares` — ready for factor construction.

---
## Step 3 — Constructing 360 Alpha Factors (the "Alpha 360")

**Script:** `data_processing_script/volume_and_price_factor_construction/3_qlib_factor_construction.ipynb`

### What is a "factor"?
A **factor** (or alpha) is a derived numerical feature that tries to capture some predictive pattern in stock prices. Think of it as a "feature" in machine learning.

### How are the 360 factors computed?
The paper uses **Microsoft Qlib's Alpha360** handler, which generates 360 factors from just 6 raw inputs: **Open, High, Low, Close, Volume, Money**.

The factors fall into several categories:

#### Category 1: Candlestick shape features (Kline descriptors)

These capture the *shape* of each day's price bar:

| Factor | Formula | What it measures |
|--------|---------|------------------|
| `KMID` | (Close − Open) / Open | Body direction & size |
| `KLEN` | (High − Low) / Open | Total range of the day |
| `KUP` | (High − max(Open,Close)) / Open | Upper shadow (selling pressure) |
| `KLOW` | (min(Open,Close) − Low) / Open | Lower shadow (buying pressure) |
| `KSFT` | (2×Close − High − Low) / Open | Shift — where price settled |

**Concrete example** for stock 000001.SZ on 2021-02-01:
```
Open=21.88, High=23.78, Low=21.60, Close=23.36

KMID  = (23.36 - 21.88) / 21.88 = 0.0676   (strong bullish bar)
KLEN  = (23.78 - 21.60) / 21.88 = 0.0996   (wide range day)
KUP   = (23.78 - 23.36) / 21.88 = 0.0192   (small upper shadow)
KLOW  = (21.88 - 21.60) / 21.88 = 0.0128   (small lower shadow)
KSFT  = (2×23.36 - 23.78 - 21.60) / 21.88 = 0.0613  (close near high)
```

#### Category 2: Price relative to historical values

These normalise today's price against recent history:

| Factor | Example | What it measures |
|--------|---------|------------------|
| `OPEN0` | Open / Close | Today's open relative to close |
| `HIGH0` | High / Close | How far above close the stock went |
| `ROC5` | Close / Close_5_days_ago − 1 | 5-day return (momentum) |
| `MA10` | Mean(Close, 10 days) / Close | 10-day moving avg vs current |
| `STD20` | Std(Close, 20 days) / Close | 20-day volatility |
| `BETA5` | Slope of log returns over 5 days | Short-term trend |

#### Category 3: Volume factors

| Factor | Example | What it measures |
|--------|---------|------------------|
| `VWAP5` | Volume-weighted avg price, 5 days | Institutional activity |
| `VSTD20` | Std(Volume, 20 days) / Volume | Volume stability |
| `VMA10` | Mean(Volume, 10 days) / Volume | Volume trend |
| `CORR10` | Correlation(Close, Volume, 10 days) | Price-volume relationship |

#### Category 4: Multi-period lookbacks

Each of the above is computed over **multiple time windows**: 5, 10, 20, 30, 60 days:

```
ROC5, ROC10, ROC20, ROC30, ROC60
MA5,  MA10,  MA20,  MA30,  MA60
STD5, STD10, STD20, STD30, STD60
... and so on
```

This generates **360 total factors** per stock per day.

### Output format
Each factor is saved as an individual CSV file where:
- **Rows** = trading dates
- **Columns** = stock codes (e.g., 000001.SZ, 000002.SZ, ...)
- **Values** = the factor value for that stock on that date

```
Alpha_360_folder/
├── KMID.csv       (shape: 700 dates × 255 stocks)
├── KLEN.csv
├── ROC5.csv
├── MA10.csv
├── ... (360 CSV files total)
```

---
## Step 4 — Factor Neutralisation

**Script:** `data_processing_script/volume_and_price_factor_construction/4_neutralization.py`

### The problem
Raw factors are contaminated by two systematic biases:

1. **Industry bias:** Banks tend to have low PE ratios, tech stocks have high PE — but that doesn't mean banks are "cheaper" in a useful sense.

2. **Market-cap (size) bias:** Large-cap stocks behave differently from small-caps. A factor that merely picks large caps isn't finding "alpha."

### How neutralisation works

For each factor, on each day, run a **cross-sectional OLS regression**:

```
Factor_value = β₀ + β₁ × log(MarketCap) + β₂ × Industry_Dummy_1 + ... + βₙ × Industry_Dummy_n + ε
```

- The **residual (ε)** is the neutralised factor — what remains after removing the explainable parts
- Then **re-standardise** to zero mean, unit variance

### Concrete example

Suppose on 2021-02-01, the raw PE factor for three banking stocks is all ~5.0 (because banks have low PEs). After neutralisation:

```
Before:  Bank_A PE = 5.1,  Bank_B PE = 4.9,  Bank_C PE = 5.0
         Tech_A PE = 45.0, Tech_B PE = 50.0

Regression explains most of the difference by industry.

After:   Bank_A residual = +0.3  (slightly higher PE *within* banking)
         Bank_B residual = -0.3  (slightly lower PE *within* banking)
         Tech_A residual = -0.5  (lower PE *within* tech)
         Tech_B residual = +0.5  (higher PE *within* tech)
```

Now the factor is truly comparing stocks **within their own industry** on a level playing field.

### Why this matters for the model
Without neutralisation, the model might simply learn "buy banks because they have low PE" — that's not useful alpha, it's just a sector bet. Neutralisation forces every feature to represent genuine stock-specific information.

---
## Step 5 — Rolling-Window Dataset Slicing

**Script:** `data_processing_script/stockformer_input_data_processing/data_Interception.py`

### What happens?
The full dataset (~6 years) is sliced into **14 overlapping subsets** in a rolling-window fashion:

```
Each subset:
├── Training:    2 years   (500 trading days)
├── Validation:  4 months  (~80 days)
└── Test:        4 months  (~80 days)
Total span:      ~2.7 years per subset
```

Windows overlap and slide forward by ~3 months:

| Subset | Train period | Test period |
|--------|-------------|-------------|
| 1 | 2018-03 → 2020-02 | 2020-07 → 2020-10 |
| 2 | 2018-05 → 2020-05 | 2020-09 → 2021-01 |
| ... | ... | ... |
| 12 | 2020-12 → 2022-12 | 2023-04 → 2023-08 |
| 14 | 2021-06 → 2023-06 | 2023-10 → 2024-01 |

### For each subset, the script:
1. Slices the `label.csv` (daily returns per stock) to the date range
2. Slices each of the 360 Alpha factor CSVs to the same range
3. Saves everything into a folder like `Stock_CN_2020-12-02_2023-08-02/`

### Why rolling windows?
- Markets change over time (regime shifts, policy changes)
- A single train/test split would only show performance for one period
- 14 overlapping windows give robust estimates of how well the model generalises across different market conditions

---
## Step 6 — Creating Model Inputs

**Script:** `data_processing_script/stockformer_input_data_processing/Stockformer_data_preprocessing_script.py`

Each subset folder contains all the inputs the model needs. Here is what each file is:

### 6a. `flow.npz` — Stock returns matrix

```python
# Shape: (num_days, 255)  e.g., (660, 255)
# Each value = daily return of that stock on that day
#
# Example row for one day:
# [0.032, -0.015, 0.008, ..., -0.003]    ← 255 stocks
#     ↑ stock_0 gained 3.2%
```

This is the **primary time-series** the model predicts.

### 6b. `trend_indicator.npz` — Binary up/down labels

```python
# Shape: (num_days, 255)
# Values: 1 if return > 0 (went up), 0 if return ≤ 0 (flat or down)
#
# Example: [1, 0, 1, ..., 0]    ← stock_0 up, stock_1 down, ...
```

Used as a **trend indicator feature** fed to the model alongside the 360 factors.

### 6c. `Alpha_360_<dates>/` — The 360 factor CSVs

These are loaded during training and **concatenated into a 3D tensor**:

```python
# Shape: (num_days, 255, 360)
#         time     stocks  factors
```

### 6d. `corr_adj.npy` — Correlation adjacency matrix

```python
# Shape: (255, 255)
# corr_adj[i, j] = Pearson correlation between return series of stock_i and stock_j
```

This captures **which stocks move together**. High positive correlation = stocks in same sector or with similar characteristics.

### 6e. `128_corr_struc2vec_adjgat.npy` — Graph embeddings

```python
# Shape: (255, 128)
# Each stock gets a 128-dimensional vector that encodes its
# structural position in the stock correlation network.
```

**How it's made:**
1. Build a graph where stocks are nodes and edges are weighted by correlation
2. Run **Struc2vec** (a graph embedding algorithm) to learn 128-d vectors
3. Stocks with similar correlation patterns get similar embeddings

**Why Struc2vec instead of Node2vec?**  
Node2vec captures *neighbourhood* similarity (who are your neighbours), while Struc2vec captures *structural role* similarity (do you behave similarly to another stock, even if you're not directly connected). Two banking stocks on different exchanges may not be correlated directly but play the same structural role.

---
## Step 7 — Wavelet Transform (Frequency Decomposition)

**Code:** `lib/Multitask_Stockformer_utils.py` → `disentangle()` function

### The key insight
Stock prices contain two types of information mixed together:

- **Low-frequency (trends):** Gradual movements driven by fundamentals, earnings, macro
- **High-frequency (noise/rapid moves):** Day-to-day fluctuations, news reactions, volatility

If you feed both to a single model, the noise can overwhelm the trend signal.

### How Discrete Wavelet Transform (DWT) works

The paper uses **Symlet-2** wavelet with **1 level of decomposition**:

```
Original return series:  [..., 0.02, -0.03, 0.05, -0.01, 0.04, ...]
                                    ↓ DWT
Low-frequency (XL):      [..., 0.01, -0.01, 0.02,  0.00, 0.02, ...]  ← smooth trend
High-frequency (XH):     [..., 0.01, -0.02, 0.03, -0.01, 0.02, ...]  ← rapid changes
```

The **sum** XL + XH ≈ original signal (it's lossless decomposition).

### In code (`disentangle` function):

```python
def disentangle(data, w='sym2', j=1):
    dwt = DWT1DForward(wave=w, J=j)     # Forward wavelet transform
    idwt = DWT1DInverse(wave=w)          # Inverse wavelet transform
    
    # Decompose into low-freq coefficients and high-freq detail coefficients
    coeff_low, coeff_high = dwt(signal)
    
    # Reconstruct low-frequency signal (zero out high-freq coefficients)
    XL = idwt((coeff_low, [zeros]))  
    
    # Reconstruct high-frequency signal (zero out low-freq coefficients)
    XH = idwt((zeros, coeff_high))
    
    return XL, XH
```

### Applied to the data

```
Input:  X  shape (batch, 20 days, 255 stocks)  — daily returns
Output: XL shape (batch, 20 days, 255 stocks)  — low-frequency component
        XH shape (batch, 20 days, 255 stocks)  — high-frequency component
```

Each 20-day window of returns is decomposed independently.

---
## Interlude — Sliding Window Samples

Before entering the model, the data is converted into sliding-window samples:

```
Given: 660 days of data, T1=20 (input window), T2=2 (prediction horizon)

Sample 0: Input = days  0..19  →  Predict days 20..21
Sample 1: Input = days  1..20  →  Predict days 21..22
Sample 2: Input = days  2..21  →  Predict days 22..23
...
Sample N: Input = days N..N+19 →  Predict days N+20..N+21
```

Each sample is a window of **20 trading days** (about 1 month), predicting the **next 2 days**.

### What each sample looks like

```python
{
    'X_low':      shape (20, 255),       # Low-freq returns
    'X_high':     shape (20, 255),       # High-freq returns
    'indicator_X': shape (20, 255),      # Trend indicator (up/down)
    'bonus_X':    shape (20, 255, 360),  # 360 Alpha factors per stock per day
    'TE':         shape (22, 2),         # Temporal encoding (day-of-week, day-of-month)
}
```

---
## Step 8 — The Stockformer Model Architecture

**Code:** `Stockformermodel/Multitask_Stockformer_models.py`

### 8a. Input Embedding

The raw inputs (1 return value + 1 trend indicator + 360 factors = **362 features** per stock per day) are embedded into a 128-dimensional space:

```
XL (low-freq) + trend_indicator + 360 factors → FeedForward(363 → 128 → 128) → embedded XL
XH (high-freq) + trend_indicator + 360 factors → FeedForward(363 → 128 → 128) → embedded XH
```

Note: Low-freq and high-freq have **separate** embedding layers, so they learn different representations.

### 8b. Temporal Embedding

The temporal position (what day of the week, what day of the month) is encoded:

```
Day of week → one-hot (5 dims)    e.g., Tuesday = [0,1,0,0,0]
Day of month → one-hot (50 dims)  e.g., Day 7 = [0,...,1,0,...] 
Concatenate → 55 dims → FeedForward(55 → 128 → 128) → TE
```

This helps the model learn patterns like "Mondays tend to be different" or "month-end rebalancing."

### 8c. Dual Encoder (×2 layers)

This is the heart of the model. **Each layer** has 4 sub-modules:

```
┌────── Low-Frequency Path ──────┐    ┌────── High-Frequency Path ─────┐
│                                │    │                                │
│  Temporal Attention            │    │  TCN (Temporal Conv Net)       │
│  (self-attention over          │    │  (dilated 1D convolutions     │
│   the 20 time steps)           │    │   to capture local patterns)  │
│         ↓                      │    │         ↓                     │
│  Spatial Attention (Low)       │    │  Spatial Attention (High)     │
│  (attention over 255 stocks)   │    │  (attention over 255 stocks)  │
│         ↓                      │    │         ↓                     │
│  Residual Connection           │    │  Residual Connection          │
│                                │    │                               │
└────────────────────────────────┘    └───────────────────────────────┘
```

**Why different processing for Low vs High?**
- **Low-freq** uses **self-attention** over time: good for capturing *long-range* trend dependencies ("the trend from 2 weeks ago is relevant today")
- **High-freq** uses **TCN** (dilated convolutions): good for capturing *local/rapid* patterns ("yesterday's spike matters")

Both paths use **causal masking** — the model can only look at *past and present*, never into the future.

#### Spatial Attention (stock-to-stock)

The spatial attention module computes attention between all 255 stocks:

```
For each time step:
  Q = Linear(stock_embeddings + graph_position)   # What am I looking for?
  K = Linear(stock_embeddings + graph_position)   # What do I have to offer?
  V = Linear(stock_embeddings + graph_position)   # What information to share?
  
  Attention = softmax(Q × K^T / √d)  # (255 × 255) attention matrix
  Output = Attention × V              # Weighted aggregation of all stocks
```

This allows the model to learn patterns like: *"When banking stocks (stock_0) move up, it's predictive for insurance stocks (stock_5)."*

The `adjgat` (Struc2vec embedding) is **added** to the stock features before computing Q, K, V — this injects graph structural information.

### 8d. Temporal Projection

After the encoder stack, the 20-day representations are projected down to 2 days:

```
Conv2d(20 → 2)  — compresses 20 time steps into 2 prediction steps
```

---
## Step 9 — Adaptive Fusion

**Code:** `adaptiveFusion` class

### The problem
We now have two representations:
- `hat_y_l`: LF path output (shape: batch × 2 days × 255 stocks × 128 dims)
- `hat_y_h`: HF path output (same shape)

How to combine them? A simple average ignores the possibility that some days need more LF info and others more HF.

### The solution: Cross-Attention Fusion

```
Query  = from Low-Frequency path (the "question asker")
Key    = from High-Frequency path (what to attend to)
Value  = from High-Frequency path (information to incorporate)

Attention = softmax(Q_lf × K_hf^T / √d)
Fused = Attention × V_hf + residual(XL)    # LF base + selective HF info
```

### Intuition
The low-frequency representation "asks" the high-frequency representation: *"Is there any short-term signal I should pay attention to?"* The attention weights automatically learn when HF information is useful (e.g., during volatile periods) and when it's noise (e.g., during calm trends).

### Key design choice
The residual connection is **only to the LF path** (`value + xl`), not both. This makes the low-frequency trend the "default" prediction, with high-frequency adjustments layered on top.

---
## Step 10 — Multi-Task Output Heads

**Code:** `StockformerOutput` class

The fused representation feeds into **two separate prediction heads**:

### Classification Head
```
FeedForward(128 → 128 → 2)   →  2 logits per stock per day
                                  class 0 = "will go down"
                                  class 1 = "will go up"
Loss: Cross-Entropy
```

### Regression Head
```
FeedForward(128 → 128 → 1)   →  1 value per stock per day
                                  = predicted return (e.g., +0.015 = +1.5%)
Loss: MAE (Mean Absolute Error)
```

### Why multi-task?
1. **Shared representations**: Both tasks force the model to learn features that are useful for *both* direction AND magnitude — this is a stronger constraint than either alone.
2. **Regularisation**: Multi-task learning acts as a natural regulariser, reducing overfitting.
3. **Practical use**: For trading, you need both — which stocks to buy (classification) AND how much to expect (regression for position sizing).

### Total loss
```
Loss = CrossEntropy(direction) + MAE(returns)
```

Both the fused output `hat_y` and the LF-only output `hat_y_l` are used for prediction, effectively creating 4 outputs (2 heads × 2 pathways).

---
## Complete Data Flow Diagram

```
RAW DATA
════════
  OHLCV + Fundamentals for ~5000 Chinese stocks
       ↓
  Filter → 255 CSI-300 stocks
       ↓
  Clean: outlier removal, forward-fill, z-score
       ↓
  Qlib Alpha360 → 360 factors/stock/day
       ↓
  Neutralise (remove industry + size bias)
       ↓
  Slice into 14 rolling subsets

MODEL INPUTS (per subset, per sample)
═════════════════════════════════════
  Daily returns → DWT → XL (20×255) + XH (20×255)
  Trend indicator → XC (20×255)
  360 Factors → bonus (20×255×360)
  Temporal encoding → TE (22×2)
  Graph embeddings → adjgat (255×128)

MODEL PIPELINE
══════════════
  ┌─ XL + XC + bonus ─→ Embed(363→128) ─┐
  │                                       │
  │  ┌─ XH + XC + bonus ─→ Embed(363→128) ─┐
  │  │                                       │
  │  │     Dual Encoder Layer 1               │
  │  │  ┌──────────┐   ┌──────────┐          │
  │  │  │ Temporal  │   │   TCN    │          │
  │  │  │ Attention │   │ (Conv1D) │          │
  │  │  │    +TE    │   │          │          │
  │  │  └────┬─────┘   └────┬─────┘          │
  │  │       ↓              ↓                 │
  │  │  Spatial Attn    Spatial Attn          │
  │  │  (Low-freq)     (High-freq)           │
  │  │  + adjgat       + adjgat              │
  │  │       ↓              ↓                 │
  │  │     Dual Encoder Layer 2 (same)        │
  │  │       ↓              ↓                 │
  │  │  Conv(20→2)     Conv(20→2)            │
  │  │       ↓              ↓                 │
  │  │       └───── Adaptive Fusion ──────┘   │
  │  │              (Cross-Attention)         │
  │  │                    ↓                   │
  │  │            Fused Features              │
  │  │         (batch × 2 × 255 × 128)       │
  │  │                    ↓                   │
  │  │     ┌──────────────┴──────────────┐   │
  │  │     ↓                             ↓   │
  │  │  Classification               Regression
  │  │  FF(128→128→2)               FF(128→128→1)
  │  │  "Up or Down?"               "How much?"
  │  │  Loss: CrossEntropy          Loss: MAE
```

---
## Model Hyperparameters

**Config:** `config/Multitask_Stock.conf`

| Parameter | Value | Meaning |
|-----------|-------|----------|
| `T1` | 20 | Look at 20 past days (~1 month) |
| `T2` | 2 | Predict 2 days ahead |
| `layers` | 2 | Number of dual-encoder blocks |
| `heads` | 1 | Number of attention heads |
| `dims` | 128 | Hidden dimension throughout |
| `samples` (sparsity) | 1.0 | No sparsity pruning |
| `wave` | sym2 | Symlet-2 wavelet family |
| `level` | 1 | 1 level of wavelet decomposition |
| `batch_size` | 12 | Training batch size |
| `learning_rate` | 0.001 | Adam optimiser |
| `max_epoch` | 100 | Maximum training epochs |
| `train/val/test` | 75/12.5/12.5% | Data split ratios |

### Parameter count

| Component | Parameters | Notes |
|-----------|-----------|-------|
| Input embeddings (LF + HF + TE) | ~152K | 363→128→128 × 2 + 55→128→128 |
| Dual encoder × 2 layers | ~660K | TCN + temporal attn + 2× spatial attn per layer |
| Adaptive fusion | ~115K | Cross-attention + feedforward |
| Classification + regression heads | ~33K | 128→128→2 + 128→128→1 |
| **Total** | **~976K** | Compact model, fits on a single GPU |

---
## Training Loop Summary

**Code:** `MultiTask_Stockformer_train.py`

```
For each epoch (100 max):
  For each batch of 12 samples:
    1. Forward pass → get 4 outputs (class_fused, class_lf, reg_fused, reg_lf)
    2. Compute loss:
       class_loss = CE(class_fused, true_class) + CE(class_lf, true_class)
       reg_loss   = MAE(reg_fused, true_return) + MAE(reg_lf, true_return)
       total_loss = class_loss + reg_loss
    3. Backpropagate and update weights (Adam)
  
  Validate on val set → track best model
  Early stopping if no improvement for N epochs
```

### Evaluation metric
- **Classification accuracy**: Percentage of (stock, day) pairs where up/down is correctly predicted
- **MAE**: Mean absolute error of predicted returns vs actual
- **RMSE**: Root mean squared error

### Backtesting (post-training)
Uses Qlib's **TopK-Dropout** strategy: rank all 255 stocks by predicted return, buy Top-K, sell Bottom-K. Evaluates Annualised Return, Sharpe Ratio, Max Drawdown, and Information Coefficient (IC).

---
## Summary: What transforms happen to a single stock's data

Let's trace a single stock (000001.SZ, Ping An Bank) through the entire pipeline:

| Step | Data | Shape per day | Example |
|------|------|---------------|--------|
| Raw | OHLCV | 5 values | O=21.88, H=23.78, L=21.60, C=23.36, V=147M |
| + Fundamentals | OHLCV + extras | ~33 columns | + PE=5.3, PB=0.88, ROE=12.4%, ... |
| After cleaning | standardised | ~10 columns | z-scored, outliers capped |
| Alpha360 | 360 factors | 360 values | KMID=0.067, ROC5=0.034, MA10=1.02, ... |
| After neutralisation | residual factors | 360 values | same 360 but industry/size adjusted |
| Model input | returns + factors | 1 + 1 + 360 = 362 | [0.032, 1, 0.067, 0.099, ...] |
| After DWT | LF & HF returns | 2 × 1 | XL=0.02, XH=0.012 |
| After embedding | latent vector | 128 dims | [0.3, -0.1, 0.7, ...] |
| After encoder | context-aware vector | 128 dims | incorporates info from all 255 stocks |
| After fusion | fused vector | 128 dims | LF trend + selective HF adjustment |
| Prediction | direction + return | 2 + 1 | P(up)=0.62, predicted return=+1.5% |

---
## Performance Summary

### Model stats
- **Total Parameters**: ~976K (lightweight)
- **Model Size**: ~3.72 MB (FP32)
- **Training time**: ~2–4 hours per subset on a single GPU

### Generalisation across rolling windows

| Subset | Test Period | Accuracy | MAE |
|--------|-----------|----------|-----|
| 1 | 2020-07 → 2020-10 | 51.66% | 0.0187 |
| 4 | 2021-03 → 2021-07 | 53.23% | 0.0207 |
| 9 | 2022-07 → 2022-11 | 54.13% | 0.0175 |
| 10 | 2022-09 → 2023-02 | 52.42% | 0.0154 |
| **12** | **2023-04 → 2023-08** | **53.79%** | **0.0145** |
| 13 | 2023-07 → 2023-11 | 55.78% | 0.0137 |

Average accuracy deviation: **±2.29%** across very different market regimes — demonstrating **good generalization**.

### Is 54% accuracy useful?
Yes! In stock markets:
- Random guess = 50%
- A consistent 54% edge, applied to 255 stocks daily, compounds into significant alpha
- The regression head matters more: correctly ranking stocks (even slightly) enables profitable long-short strategies

---
## Key File Reference

| Purpose | File | Location |
|---------|------|----------|
| Data consolidation | `1_stock_data_consolidation.ipynb` | `data_processing_script/volume_and_price_factor_construction/` |
| Stock pool cleaning | `2_data_preprocessing.ipynb` | same folder |
| Factor construction | `3_qlib_factor_construction.ipynb` | same folder |
| Neutralisation | `4_neutralization.py` | same folder |
| Factor verification | `5_factor_verification.ipynb` | same folder |
| Time-period slicing | `data_Interception.py` | `data_processing_script/stockformer_input_data_processing/` |
| Creates flow.npz + graph | `Stockformer_data_preprocessing_script.py` | same folder |
| Model architecture | `Multitask_Stockformer_models.py` | `Stockformermodel/` |
| Data loading + DWT | `Multitask_Stockformer_utils.py` | `lib/` |
| Training loop | `MultiTask_Stockformer_train.py` | root |
| Configuration | `Multitask_Stock.conf` | `config/` |

---

### References

1. **Paper**: Ma, Bohan; Xue, Yushan; Lu, Yuan & Chen, Jing. (2025). *Stockformer*. Expert Systems with Applications, 273, 126803.  
2. **Qlib (Alpha360)**: https://github.com/microsoft/qlib  
3. **Struc2vec**: https://github.com/shenweichen/GraphEmbedding  
4. **PyTorch Wavelets**: https://github.com/fbcotter/pytorch_wavelets